In [1]:
import os
os.environ["GROQ_API_KEY"]="your api key"

In [2]:
!pip install -q langchain-groq langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [3]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# creating a custom tool
@tool
def multiply(a: int, b: int)->int:
  """Given 2 numbers a and b this tool returns their product"""
  return a*b

In [7]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [8]:
print(multiply.invoke({"a": 3, "b": 4}))

12


In [9]:
multiply.name

'multiply'

In [10]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [11]:
# now we will create a llm
LLM_Model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.5)

In [15]:
# binding llm with this tool
LLM_with_tool_bind = LLM_Model.bind_tools([multiply])
# if there are more tools then just put comma and give your tool name
# also not all llm capable of binding with tools


In [17]:
# lets ask it normally
LLM_with_tool_bind.invoke("hi how are you")

AIMessage(content="Hello! I'm doing great, thank you for asking. How can I assist you today?", additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says "hi how are you". Simple greeting. Should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 133, 'total_tokens': 185, 'completion_time': 0.108728889, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.005033632, 'prompt_tokens_details': None, 'queue_time': 0.06753876, 'total_time': 0.113762521}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_19b184c447', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ee945-647c-7110-a667-c3603bc45f3a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 52, 'total_tokens': 185, 'output_token_details': {'reasoning': 25}})

In [31]:
Query = HumanMessage("what is the multiplication of 34 and 1234?")

In [32]:
message=[Query]

In [33]:
result = LLM_with_tool_bind.invoke(message)

In [34]:
message.append(result)

In [37]:
LLM_with_tool_bind.invoke(message).content

'The product of 34 and 1234 is **41,956**.'

In [38]:
# lets ask it now about the multiplicaton
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 34, 'b': 1234},
  'id': 'fc_faeed0c9-88d8-46f6-9215-cbed6d5872f7',
  'type': 'tool_call'}]

In [39]:
result.tool_calls[0]['args']

{'a': 34, 'b': 1234}

In [40]:
multiply.invoke(result.tool_calls[0]['args'])

41956

In [41]:
# what is we send whole tool call in it
multiply.invoke({'name': 'multiply',
  'args': {'a': 34, 'b': 1234},
  'id': 'fc_e28c418f-eec9-4359-95d8-7822dd892a13',
  'type': 'tool_call'})

ToolMessage(content='41956', name='multiply', tool_call_id='fc_e28c418f-eec9-4359-95d8-7822dd892a13')